In [0]:
env_name = 'INTERNAL_retrieval_augmented_generation_v1'
env_lang = 'PYTHON'
import dataiku

# Scope: Current Project
client = dataiku.api_client()

code_env_handle = client.get_code_env(env_lang, env_name)

# https://developer.dataiku.com/latest/api-reference/python/code-envs.html#dataikuapi.dss.admin.DSSCodeEnv.list_usages
# code env's list_usages does not take any parameters
code_env_handle.list_usages()

In [0]:
import dataiku
from dataikuapi.utils import DataikuException

# --------------------------------------------------------------------------------
# Configuration & Constants
# --------------------------------------------------------------------------------

# List of known Core Agent Tool types in Dataiku DSS v14.2+
# These are internal identifiers for the native managed tools.
CORE_AGENT_TOOL_TYPES = [
    "ClassicalPredictionModelPredict",  # Model Predict
    "VectorStoreSearch",                # Knowledge Bank Search
    "LLMMeshLLMQuery",                  # Query an LLM
    "DataikuReporter",                  # Send Message / Reporter
    "InlinePython",                     # Inline Python Code
    "DatasetRowAppend",                 # Dataset Append
    "DatasetLookup",                    # Dataset Lookup
    "SQLQuery",                         # SQL Query
    "GenericStdioMCPClient",            # Local MCP
    "GenericHttpMCPClient"              # Remote MCP
]

def get_plugin_agent_tool_types(client):
    """
    Dynamically fetches Agent Tool types defined in installed plugins.
    
    Args:
        client (DSSClient): The Dataiku API client.
        
    Returns:
        list: A list of string identifiers for plugin-based agent tools
              formatted as 'Custom_<PluginID>_<ComponentID>'.
    """
    plugin_tool_types = []
    
    try:
        # Retrieve all installed plugins
        plugins = client.list_plugins()
        
        for plugin in plugins:
            # Plugins contain a list of components (recipes, webapps, tools, etc.)
            # We filter for components specifically related to Agent Tools.
            # Note: The exact 'type' identifier for agent tools in plugin.json is typically 'AGENT_TOOL'
            # or inferred from the component definition structure.
            print(plugin)
            
            # Use safe get() to avoid key errors if metadata is missing
            components = plugin.get('components', [])
            
            for component in components:
                # Check if the component is an Agent Tool
                # We check for the explicit type or if it's categorized under agent tools
                comp_type = component.get('type', '')
                
                if comp_type == 'AGENT_TOOL':
                    plugin_id = plugin.get('id')
                    comp_id = component.get('id')
                    
                    # Construct the unique type identifier
                    # Convention: Custom_<PluginID>_<ComponentID>
                    full_type_id = f"Custom_{plugin_id}_{comp_id}"
                    plugin_tool_types.append(full_type_id)
                    
    except DataikuException as e:
        print(f"Error fetching plugin list: {e}")
        # In a strict idempotent run, we might choose to raise or return empty
        # Here we return empty to allow the script to proceed with Core types
        return []

    return plugin_tool_types

def get_all_available_agent_tool_types():
    """
    Aggregates Core and Plugin agent tool types into a single unique list.
    """
    # 1. Setup Client (Assumes local execution as Admin)
    client = dataiku.api_client()
    
    # 2. Get Core Types (Static definition based on DSS 14.2+)
    # In a reverse-engineering context, these could potentially be fetched 
    # via internal API: client._perform_json("GET", "/generative-ai/agent-tools/types")
    # but the static list is safer and strictly compliant with public APIs.
    all_types = set(CORE_AGENT_TOOL_TYPES)
    
    # 3. Get Plugin Types (Dynamic fetch)
    plugin_types = get_plugin_agent_tool_types(client)
    all_types.update(plugin_types)
    
    # 4. Return sorted list for consistency
    return sorted(list(all_types))

# --------------------------------------------------------------------------------
# Main Execution
# --------------------------------------------------------------------------------

if __name__ == "__main__":
    try:
        available_tools = get_all_available_agent_tool_types()
        
        # Output strictly the list as requested
        print(available_tools)
        
    except Exception as e:
        print(f"Failed to retrieve Agent Tool types: {e}")